In [1]:
from dotenv import load_dotenv
import os
from anthropic import Anthropic

In [2]:
if os.getenv("ANTHROPIC_API_KEY") is not None:
    print("API key is set")
else:
    print("API key is not set")

API key is set


In [3]:
client = Anthropic()

In [4]:
def add_user_message(messeges, content):
    messeges.append({"role": "user", "content": content})

def add_assistant_message(messeges, content):
    messeges.append({"role": "assistant", "content": content})

def chat_with_claude(messages,stop_sequences=None):
    if stop_sequences is None:
        args={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 1024,
            "messages": messages
        }
    else:
        args={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 1024,
            "messages": messages,
            "stop_sequences": stop_sequences
        }
    response = client.messages.create(**args)
    return response.content[0].text

In [7]:
from anthropic.types import ToolParam

In [8]:
def get_current_date_time(date_format="%Y-%m-%d %H:%M:%S"):
    from datetime import datetime
    if not date_format:
        raise ValueError("Date format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_date_time_schema = ToolParam({
  "name": "get_current_date_time",
  "description": "Returns the current date and time formatted according to the specified format string.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string used to format the date and time. For example, '%Y-%m-%d %H:%M:%S' produces '2026-04-19 14:30:00'. Cannot be empty.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": ["date_format"]
  }
})

In [10]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=messages,
    tools=[get_current_date_time_schema],
)

In [16]:
response

Message(id='msg_01AHcSD9KBCskgCoEhLkgmUN', container=None, content=[TextBlock(citations=None, text="I'll get the current time formatted as HH:MM:SS for you.", type='text'), ToolUseBlock(id='toolu_01DzRgQxFNefvjHG4fbMn3NZ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_date_time', type='tool_use')], model='claude-sonnet-4-20250514', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=473, output_tokens=82, server_tool_use=None, service_tier='standard'))

In [15]:
response.content[0]

TextBlock(citations=None, text="I'll get the current time formatted as HH:MM:SS for you.", type='text')